# 학급 테이블 전처리 수행

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [3]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_group`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

print(df.head())

     id  grade  class_num  school_id
0     1      1          1          1
1   227      1         20          1
2   660      1          7          1
3  1184      1          5          1
4  1191      1          8          1


## 결측치 확인 및 데이터 정보 확인

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 84515 entries, 0 to 84514
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         84515 non-null  Int64
 1   grade      84515 non-null  Int64
 2   class_num  84515 non-null  Int64
 3   school_id  84515 non-null  Int64
dtypes: Int64(4)
memory usage: 2.9 MB


* 결측치(Missing Value) 없음

* 전체 컬럼 수치형(Numeric) 데이터로 구성

## 중복값 체크

In [5]:
# 중복값 체크 
df.duplicated().sum()

df['id'].duplicated().sum()

# 학교, 학년, 반 중복 확인
df[df[['grade', 'class_num', 'school_id']].duplicated(keep=False)]

,id,grade,class_num,school_id
13835,58711,1,4,4383
13838,76464,1,4,4383
19455,58724,1,1,5899
19458,77120,1,1,5899
54998,80809,2,20,5900
54999,82115,2,20,5900
75813,75762,3,1,4383
75815,79806,3,1,4383
84330,58927,3,1,5900
84331,73234,3,1,5900


* 완전 중복행 없음: 전체 행 기준 중복값 및 식별자(ID) 중복 없음

* 반 정보 중복 허용: 학교·학년·반 조합이 동일한 데이터가 존재하나, 학년도(연도) 변경에 따른 신규 학급 생성 가능성을 고려하여 정상 데이터로 처리

## 이상치 판별

In [7]:
# 이상치 판별
df.describe()

,id,grade,class_num,school_id
count,84515.0,84515.0,84515.0,84515.0
mean,42279.376087,2.113992,5.318606,3079.496468
std,24400.96263,0.754896,4.089901,1675.036093
min,1.0,1.0,0.0,1.0
25%,21147.5,2.0,2.0,1752.0
50%,42279.0,2.0,4.0,3023.0
75%,63411.5,3.0,7.0,4620.0
max,84546.0,20.0,41.0,5965.0


* 학년(grade) 이상치: '20학년' 등 비현실적인 학년값 식별 → 오기입 여부 확인 및 처리 기준 수립 필요

* 반(class_num) 이상치: '41반' 등 통상적인 학급 범위를 벗어난 값 존재 → 특수 학급/가상 코드 여부 확인 필요

### 학년 이상치 확인
* 3학년을 넘어가는 학급 확인

In [24]:
# grade 이상치 확인
df[df['grade'] > 3]

,id,grade,class_num,school_id
84513,32644,4,10,4658
84514,61647,20,7,3867


In [26]:
# 학 학교의 학년 및 반 배치 분포 확인
df[(df['school_id'] == 4658) | (df['school_id'] == 3867)].sort_values(by=['school_id', 'grade', 'class_num'])

,id,grade,class_num,school_id
42085,41808,2,1,3867
42090,55675,2,2,3867
42088,54817,2,3,3867
42089,54976,2,4,3867
42086,43476,2,5,3867
42087,47050,2,6,3867
42084,39224,2,7,3867
42093,84108,2,10,3867
42092,62159,2,16,3867
42091,56063,2,20,3867


* 학년 표기 오류 판별 한계: 학교별 학년 분포만으로는 해당 학년 데이터의 실제 오류 여부를 확정하기 어려움

In [28]:
# 학년 이상 그룹의 학생 수 조회

temp_sql = f"""
    SELECT group_id, COUNT(*)
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_user`
    WHERE group_id IN (32644, 61647)
    GROUP BY group_id
"""

# 판다스 데이터프레임으로 변환
temp_df = client.query(temp_sql).to_dataframe()

temp_df.head()

,group_id,f0_


* 구성 데이터(표본)가 존재하지 않는 해당 2개 그룹 삭제 진행

### 반 이상치 확인

In [14]:
# 반별 개수 (반 번호 순서 정렬)
df['class_num'].value_counts().sort_index()

class_num
0         2
1     12125
2     11077
3     10276
4      9506
5      8578
6      7555
7      6390
8      5294
9      3799
10     3067
11     1600
12      998
13      506
14      311
15      216
16      152
17      214
18      217
19      177
20     2454
41        1
Name: count, dtype: Int64

* 희소 학급(0반·41반): 빈도수가 극히 적어 입력 오류 등 이상치일 가능성이 높음

* 분포 역전 학급(20반): 후순위 반 번호일수록 빈도가 감소하는 일반적인 우하향 패턴과 달리 빈도가 급증하여 비정상적 분포를 형성함

In [16]:
df[(df['class_num'] == 0) | (df['class_num'] == 41)]

,id,grade,class_num,school_id
2100,70394,1,41,736
29164,62651,2,0,1820
44355,62259,2,0,4348


In [29]:
# 반 이상 그룹의 학생 수 조회

temp_sql = f"""
    SELECT group_id, COUNT(*)
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_user`
    WHERE group_id IN (70394, 62651, 62259)
    GROUP BY group_id
"""

# 판다스 데이터프레임으로 변환
temp_df = client.query(temp_sql).to_dataframe()

temp_df.head()

,group_id,f0_
0,62651,1
1,62259,1


In [ ]:
# 반 이상 그룹의 학생 조회

temp_sql = f"""
    SELECT * FROM {PROJECT_ID}.{DATA_SET}.accounts_user
    WHERE group_id IN (62651, 62259);
"""

# 판다스 데이터프레임으로 변환
temp_df = client.query(temp_sql).to_dataframe()

temp_df.head()

,id,is_superuser,is_staff,gender,point,friend_id_list,is_push_on,created_at,block_user_id_list,hide_user_id_list,ban_status,report_count,alarm_count,pending_chat,pending_votes,group_id
0,1007923,0,0,M,712,"[1114348, 960589, 1480173, 1428913, 939604, 89...",1,2023-05-08 12:15:48.935126+00:00,[],[],N,0,2,0,8,62651
1,1291095,0,0,F,314,"[1246371, 1242117, 1282836, 1231928, 1213467, ...",1,2023-05-15 12:26:16.121182+00:00,[],[],N,0,1,0,1,62259


* 결측 그룹 정제: 구성원이 0명인 '70394' 그룹은 유효 데이터가 아니므로 영구 삭제 처리함.

* 극소 표본 그룹(2개) 유지 결정 사유:

* 정상 유저 확인: 해당 그룹의 유저(각 1명)는 친구 목록 및 활동 이력이 확인된 유효 계정임.

* 소속 유지 필요성: 향후 소셜 네트워크(친구 관계) 기반 그룹 재분류 전까지 임시 소속 유지 필요.

* 분석 영향도 미미: 반(그룹) 단위 분석 지표는 주로 집단 가입 건에 한정되어 있어, 해당 단일 구성원 그룹을 유지하더라도 전체 분석 결과에 미치는 왜곡이 미미하다고 판단하여 현 상태 유지.

In [33]:
# 20반 중 학생 수가 0인 곳이 있는지 파악

temp_sql = f"""
    SELECT u.group_id
            , COUNT(*) AS cnt
    FROM {PROJECT_ID}.{DATA_SET}.accounts_user AS u
    INNER JOIN {PROJECT_ID}.{DATA_SET}.accounts_group AS g
    ON u.group_id = g.id
    WHERE g.class_num = 20
    GROUP BY u.group_id
    HAVING cnt = 0
"""

# 판다스 데이터프레임으로 변환
temp_df = client.query(temp_sql).to_dataframe()

temp_df.head()

,group_id,cnt


In [ ]:
# 20반 중 학생 수가 1인 곳이 있는지 파악

temp_sql = f"""
WITH class_summary AS (
    SELECT 
        u.group_id,
        COUNT(*) AS student_count
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_user` AS u
    INNER JOIN `{PROJECT_ID}.{DATA_SET}.accounts_group` AS g
        ON u.group_id = g.id
    WHERE g.class_num = 20
    GROUP BY u.group_id
)
SELECT 
    COUNT(*) AS total_class_count,
    COUNTIF(student_count = 1) AS single_student_class_count,
    ROUND(COUNTIF(student_count = 1) / COUNT(*) * 100, 2) AS ratio_pct
FROM class_summary;
"""

# 판다스 데이터프레임으로 변환
temp_df = client.query(temp_sql).to_dataframe()

temp_df.head()

,total_class_count,single_student_class_count,ratio_pct
0,2015,1226,60.84


* 학급 표본 유효성 검증: 전체 20개 학급 모두 최소 1명 이상의 학생이 배정되어 결측 학급(0명)은 존재하지 않음.

* 단일 학생 학급 유지 사유: 전체의 절반 이상이 학생 수 1명인 학급으로 구성되어 있으나, 정상 유저 확인 및 반 단위 분석에 미치는 왜곡이 제한적인 점을 고려하여 해당 학급들은 현행 유지하기로 결정함.

In [36]:
# 데이터 삭제 진행
# 학년 이상 그룹 (32644, 61647)
# 반 이상 그룹 (70394)
del_sql = f"""
        DELETE FROM `{PROJECT_ID}.{DATA_SET}.accounts_group`
        WHERE id IN (32644, 61647, 70394)
"""

# 1. 삭제 전 사전 검증
check_sql = f"""
    SELECT 
        g.id AS group_id,
        COUNT(u.id) AS user_cnt
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_group` AS g
    LEFT JOIN `{PROJECT_ID}.{DATA_SET}.accounts_user` AS u
        ON g.id = u.group_id
    WHERE g.id IN (32644, 61647, 70394)
    GROUP BY g.id
"""

check_df = client.query(check_sql).to_dataframe()
print("=== 삭제 대상 사전 검증 ===")
print(check_df)

# 유저가 1명 이상 남아있는 그룹이 있는지 확인
active_groups = check_df[check_df['user_cnt'] > 0]

if not active_groups.empty:
    print(f"\n[경고] 연결된 유저가 존재하는 그룹이 있습니다:\n{active_groups}")
    print("삭제를 중단합니다. 유저 매핑을 먼저 정리해 주세요.")
else:
    print("\n연결된 유저가 0명이므로 안전하게 삭제를 진행합니다.")
    query_job = client.query(del_sql)
    query_job.result()
    print(f"삭제 완료! 삭제된 행 수: {query_job.num_dml_affected_rows}")

=== 삭제 대상 사전 검증 ===
   group_id  user_cnt
0     70394         0
1     32644         0
2     61647         0

연결된 유저가 0명이므로 안전하게 삭제를 진행합니다.
삭제 완료! 삭제된 행 수: 3
